In [3]:
import json
from tqdm.auto import tqdm
from elasticsearch import Elasticsearch

# 1. Cargamos nuestros documentos que ahora ya tienen el campo 'id'
with open('documents-with-ids.json', 'rt') as f_in:
    documents = json.load(f_in)

# 2. Nos conectamos a Elasticsearch
es_client = Elasticsearch('http://localhost:9200') 

# 3. Definimos la estructura de nuestra base de datos (Nota que ahora incluimos "id")
index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "keyword"},
            "id": {"type": "keyword"}, # ¡Nuestro nuevo campo!
        }
    }
}

index_name = "course-questions"

# 4. Borramos el índice si ya existía y creamos uno nuevo limpio
es_client.indices.delete(index=index_name, ignore_unavailable=True)
es_client.indices.create(index=index_name, body=index_settings)

print("Subiendo documentos a Elasticsearch...")

# 5. Subimos los documentos uno por uno
for doc in tqdm(documents):
    es_client.index(index=index_name, document=doc)

print("¡Indexación completada!")

Subiendo documentos a Elasticsearch...


  0%|          | 0/948 [00:00<?, ?it/s]

¡Indexación completada!


In [4]:
import pandas as pd
from tqdm.auto import tqdm

# 1. Función de búsqueda en Elasticsearch
def elastic_search(query, course):
    search_query = {
        "size": 5,
        "query": {
            "bool": {
                "must": {
                    "multi_match": {
                        "query": query,
                        "fields": ["question^3", "text", "section"],
                        "type": "best_fields"
                    }
                },
                "filter": {
                    "term": {
                        "course": course
                    }
                }
            }
        }
    }

    response = es_client.search(index=index_name, body=search_query)
    
    result_docs = []
    for hit in response['hits']['hits']:
        result_docs.append(hit['_source'])
    
    return result_docs

# 2. Cargamos nuestro Ground Truth (La hoja de respuestas)
df_ground_truth = pd.read_csv('ground-truth-data.csv')
ground_truth = df_ground_truth.to_dict(orient='records')

print("Haciendo las consultas a Elasticsearch (Esto tomará un minuto o dos)...")

# 3. Ejecutamos el examen
relevance_total = []

for q in tqdm(ground_truth):
    doc_id = q['document'] # El ID que sabemos que es correcto
    
    # Hacemos la búsqueda
    results = elastic_search(query=q['question'], course=q['course'])
    
    # Comparamos los IDs devueltos con el ID correcto
    relevance = [d['id'] == doc_id for d in results]
    relevance_total.append(relevance)

print("\n¡Examen finalizado!")
print(f"Ejemplo de resultado para la primera pregunta: {relevance_total[0]}")

Haciendo las consultas a Elasticsearch (Esto tomará un minuto o dos)...


  0%|          | 0/4622 [00:00<?, ?it/s]


¡Examen finalizado!
Ejemplo de resultado para la primera pregunta: [True, False, False, False, False]


In [5]:
# 1. Función para calcular el Hit Rate (Tasa de Éxito)
def hit_rate(relevance_total):
    cnt = 0
    for line in relevance_total:
        if True in line: # Si hay al menos un True en el Top 5, cuenta como éxito
            cnt = cnt + 1
    return cnt / len(relevance_total)

# 2. Función para calcular el MRR (Rango Recíproco Medio)
def mrr(relevance_total):
    total_score = 0.0
    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == True:
                # Si está en la posición 0, suma 1/1. Si está en la 1, suma 1/2, etc.
                total_score = total_score + 1 / (rank + 1)
    return total_score / len(relevance_total)

# 3. ¡Calculamos las notas para Elasticsearch!
print("--- RESULTADOS DE ELASTICSEARCH ---")
print(f"Hit Rate: {hit_rate(relevance_total):.4f}")
print(f"MRR:      {mrr(relevance_total):.4f}")

--- RESULTADOS DE ELASTICSEARCH ---
Hit Rate: 0.7397
MRR:      0.6031


In [6]:
import minsearch

# 1. Configuramos y llenamos el índice de minsearch (Nota: agregamos el campo 'id')
index = minsearch.Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["course", "id"] 
)
index.fit(documents)

# 2. Función de búsqueda para minsearch (con los pesos/boosts que ya conocíamos)
def minsearch_search(query, course):
    boost = {'question': 3.0, 'section': 0.5}
    results = index.search(
        query=query,
        filter_dict={'course': course},
        boost_dict=boost,
        num_results=5
    )
    return results

# 3. La nueva función general de Evaluación
def evaluate(ground_truth, search_function):
    relevance_total = []
    for q in tqdm(ground_truth):
        doc_id = q['document']
        # Usamos la función de búsqueda que nos pasen como parámetro
        results = search_function(q)
        relevance = [d['id'] == doc_id for d in results]
        relevance_total.append(relevance)

    return {
        'hit_rate': hit_rate(relevance_total),
        'mrr': mrr(relevance_total)
    }

# 4. ¡El Gran Final! Evaluamos minsearch y lo mostramos
print("Evaluando minsearch...")
resultados_minsearch = evaluate(
    ground_truth, 
    lambda q: minsearch_search(q['question'], q['course']) # Pasamos minsearch_search como parámetro
)

print("\n--- RESULTADOS MINSEARCH ---")
print(f"Hit Rate: {resultados_minsearch['hit_rate']:.4f}")
print(f"MRR:      {resultados_minsearch['mrr']:.4f}")

Evaluando minsearch...


  0%|          | 0/4622 [00:00<?, ?it/s]


--- RESULTADOS MINSEARCH ---
Hit Rate: 0.7720
MRR:      0.6615
